<a href="https://colab.research.google.com/github/Eratofee/ChromaTone/blob/main/ChromaTone_AI_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ChromaTone: build an image-to-music instrument
## AI I Lab · 16–18 November 2026 · Student notebook · by Dr. Esther Fee Reinhardt

How can a drawing become music? In this lab you will explore a working instrument, investigate its rules and shape its musical behaviour. **You do not need to write new code.** Run the supplied cells, change clearly marked values, listen and document your decisions.

Work from top to bottom. Pause at each portfolio task before revealing/running the next implementation. Text cells are editable; copy your completed sections and saved evidence into your portfolio.

**Monday:** explore modality translation, parameters, training material and Markov chains. **Tuesday:** experiment, propose mappings and optionally add motifs. **Wednesday:** rehearse, perform and document your contribution.

Pair work: share decisions, screen and sound with your remote partner. Each student writes their own reflections and identifies their contribution.

**Environment:** Google Colab with a Python runtime and browser audio. The notebook contains its training data and all project code: no repository download, GitHub account or external project files are needed. Colab and initial package installation still require internet access. This version uses a digital canvas and browser sound; camera input and external MIDI output are future extensions.

Run individual cells during the workshop rather than “Run all”, so you can stop for research. To restart, stop the live instrument first, restart the runtime and run code cells in order. Download your edited notebook regularly using File → Download → Download .ipynb.


## 1. From one modality to another


### Portfolio task — Modality translation

Before running the next code cell, research the following question. Write **120–180 words** in your own words (approximately one third to half an A4 page, depending on layout). Record at least one source with author/organisation, title, URL and access date. Distinguish a source’s claim from your own interpretation.

What is a modality? Explain one image-to-sound example and one different cross-modal example. Which properties can be measured, and which relationships must a designer choose? Explain why a colour-to-note rule is not automatically a learned AI model.

Double-click the following text cell to edit it. Keep your completed text for your portfolio.


**Your response:**

[Write your researched explanation here.]

**Sources:**

[Add your references here.]

**Your own question or design idea:**

[Write here.]


### The instrument’s route
Drawing → measured colours, brightness and pointer movement → musical settings → a trained second-order Markov chain → generated pitches → sound.

The image-to-music mappings are hand-designed. The Markov chain estimates possible continuations from examples. There is no neural network and no semantic understanding of the image.

### Prepare the tools
Run this once. The packages are general-purpose tools; all project data is already below.


In [1]:
%pip -q install numpy pandas pillow matplotlib

import base64, io, random, json
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from matplotlib.colors import rgb_to_hsv
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from IPython.display import Audio, HTML, JSON as DisplayJSON, display

print("Tools ready. Continue to section 2.")


Tools ready. Continue to section 2.


## 2. Parameters are artistic decisions


### Portfolio task — Choose your mappings

Before running the next code cell, research the following question. Write again **120–180 words** in your own words. Record at least one source with author/organisation, title, URL and access date. Distinguish a source’s claim from your own interpretation.

Research parameter mapping in sonification or interactive music. Explain the difference between an input feature and an output parameter. Read the current mappings below. Propose three additional mappings (for example spatial density → register), justify their artistic purpose and name one limitation.

Double-click the following text cell to edit it. Keep your completed text for your portfolio.


**Your response:**

[Write your researched explanation here.]

**Sources:**

[Add your references here.]

**Your own question or design idea:**

[Write here.]


### Current mappings
| Input or control | Musical consequence | Limitation to investigate |
|---|---|---|
| Most common recognised colour | Tonal centre / transposition | Ties select the first pitch class; black is excluded |
| Selected brush colour (optional) | Overrides the tonal centre | Uses the selected colour, not the whole picture |
| Mean brightness of the entire canvas | Major/minor training subset | The black background counts; sparse drawings tend to select minor |
| Recent vertical pointer motion | Ascending, descending, oscillating or horizontal subset | Labels describe training material, not guaranteed output contours |
| Recent pointer speed | Note duration | Speed is sampled from recent pointer events, not inferred from still images |
| Motif pitch sequences | Possible Markov transitions | No original rhythm, dynamics or harmony is learned |
| Brush size | Changes the painted area | No direct loudness mapping in this version |

Colour mappings are design choices, not universal equivalences. Black and near-black pixels do not contribute to colour counts; neutral greys can still enter a hue bin. A blank canvas defaults to C and minor.

**Try later:** change only one value below at a time. Stop live playback, rerun this cell, then restart playback. For update interval changes rerun the canvas cell too.


In [2]:
# EDIT THESE VALUES, then rerun this cell.
BRIGHTNESS_THRESHOLD = 127     # 0–255; higher values make minor more likely
MODE_OVERRIDE = "auto"         # "auto", "maj", or "min"
NOTES_PER_PHRASE = 6           # try 4, 6, 8 or 12
UPDATE_SECONDS = 2.0           # desired minimum time between image requests
DURATION_MULTIPLIER = 1.0      # try 0.7, 1.0 or 1.4
REGISTER_SHIFT = 0            # semitones: try -12, 0 or 12

PITCH_CLASSES = ['C', 'Db', 'D', 'Eb', 'E', 'F', 'Gb', 'G', 'Ab', 'A', 'Bb', 'B']
DIRECTIONS = {0: 'ascending', 1: 'descending', 2: 'oscillating', 3: 'horizontal'}
COLOR_RANGES = {
 'red': [(0,10),(166,180)],
 'orange': [(11,23)],
 'yellow': [(24,37)],
 'spring_green': [(38,55)],
 'green': [(56,68)],
 'turquoise': [(69,82)],
 'cyan': [(83,99)],
 'ocean': [(100,113)],
 'blue': [(114,127)],
 'violet': [(128,145)],
 'magenta': [(146,165)]}
COLOR_TO_NOTE = {'white':'C',
                 'ocean':'Db',
                 'turquoise':'D',
                 'orange':'Eb',
                 'yellow':'E',
                 'green':'F',
                 'spring_green':'Gb',
                 'blue':'G',
                 'cyan':'Ab',
                 'red':'A',
                 'magenta':'Bb',
                 'violet':'B'}
assert MODE_OVERRIDE in ('auto','maj','min')
assert 0 <= BRIGHTNESS_THRESHOLD <= 255
assert isinstance(NOTES_PER_PHRASE, int) and 1 <= NOTES_PER_PHRASE <= 32
assert 0.2 <= UPDATE_SECONDS <= 10 and 0.2 <= DURATION_MULTIPLIER <= 3
assert -24 <= REGISTER_SHIFT <= 24
print(pd.DataFrame(COLOR_TO_NOTE.items(), columns=['Colour','Pitch class']).to_string(index=False))


      Colour Pitch class
       white           C
       ocean          Db
   turquoise           D
      orange          Eb
      yellow           E
       green           F
spring_green          Gb
        blue           G
        cyan          Ab
         red           A
     magenta          Bb
      violet           B


## 3. Training material: inspect what the model can learn


### Portfolio task — Training data and musical possibilities

Before running the next code cell, research the following question. Write **120–180 words** in your own words. Record at least one source with author/organisation, title, URL and access date. Distinguish a source’s claim from your own interpretation.

What can a model learn from ordered MIDI pitch numbers, and what cannot be recovered without timing or velocity? Explain how the choice of training examples constrains a generative system. Suggest a musical quality missing from this dataset that you would want to add.

Double-click the following text cell to edit it. Keep your completed text for your portfolio.


**Your response:**

[Write your researched explanation here.]

**Sources:**

[Add your references here.]

**Your own question or design idea:**

[Write here.]


### Embedded pitch data
The complete supplied pitch table is included below. **48 major/minor motifs are used by the instrument.** Eight additional motifs labelled `eas` are retained for transparency but are not selected by the major/minor controls. We do not infer a precise scale definition from that label.

Each list contains MIDI pitches in order (60 = C4). Original note lengths and velocities were not included in the source table. The notation exercise therefore uses equal quarter notes as a display convention.

Data provenance: embedded snapshot of the supplied ChromaTone project’s `motifs_df/midi_motives.csv` and `initial_notes.json`, retrieved 15 November 2026. No source connection is made when you run this notebook.


In [3]:
TRAINING_MOTIFS = [{'id': 'M01',
  'name': 'min Asc S 1',
  'direction': 0,
  'scale': 'min',
  'notes': [48, 55, 60, 62, 63, 67, 72, 79]},
 {'id': 'M02',
  'name': 'Eas Asc 2',
  'direction': 0,
  'scale': 'eas',
  'notes': [55, 56, 53, 59, 60, 56, 62, 65, 59, 67, 68]},
 {'id': 'M03',
  'name': 'Maj Asc 5',
  'direction': 0,
  'scale': 'maj',
  'notes': [52, 53, 55, 60, 64, 65, 67, 72]},
 {'id': 'M04',
  'name': 'Eas Osc 2',
  'direction': 2,
  'scale': 'eas',
  'notes': [66, 63, 56, 59, 67, 55, 60, 59]},
 {'id': 'M05',
  'name': 'min Des S 2',
  'direction': 1,
  'scale': 'min',
  'notes': [75, 74, 72, 67, 63, 62, 60, 55]},
 {'id': 'M06',
  'name': 'min Des S 1',
  'direction': 1,
  'scale': 'min',
  'notes': [79, 75, 74, 72, 67, 63, 62, 60]},
 {'id': 'M07',
  'name': 'Eas Hor 1',
  'direction': 3,
  'scale': 'eas',
  'notes': [55, 56, 59, 55, 56, 59, 56, 55]},
 {'id': 'M08',
  'name': 'Maj Des 5',
  'direction': 1,
  'scale': 'maj',
  'notes': [79, 74, 72, 71, 67, 62, 60, 59]},
 {'id': 'M09',
  'name': 'Maj Osc 2',
  'direction': 2,
  'scale': 'maj',
  'notes': [53, 60, 55, 60, 57, 60, 55, 53]},
 {'id': 'M10',
  'name': 'Eas Des 1',
  'direction': 1,
  'scale': 'eas',
  'notes': [79, 78, 75, 74, 72, 71, 68, 67]},
 {'id': 'M11',
  'name': 'min Osc 2',
  'direction': 2,
  'scale': 'min',
  'notes': [56, 65, 55, 63, 56, 65, 58, 56]},
 {'id': 'M12',
  'name': 'Maj Des 2',
  'direction': 1,
  'scale': 'maj',
  'notes': [74, 72, 69, 66, 62, 60, 57, 54]},
 {'id': 'M13',
  'name': 'Maj Hor 1',
  'direction': 3,
  'scale': 'maj',
  'notes': [60, 60, 62, 60, 64, 60, 62, 60]},
 {'id': 'M14',
  'name': 'min Des S 5',
  'direction': 1,
  'scale': 'min',
  'notes': [80, 79, 77, 72, 68, 67, 65, 60]},
 {'id': 'M15',
  'name': 'Maj Hor 3',
  'direction': 3,
  'scale': 'maj',
  'notes': [60, 60, 59, 60, 64, 60, 59, 60]},
 {'id': 'M16',
  'name': 'Maj Asc 2',
  'direction': 0,
  'scale': 'maj',
  'notes': [48, 57, 53, 55, 60, 65, 67, 72]},
 {'id': 'M17',
  'name': 'Eas Des 2',
  'direction': 1,
  'scale': 'eas',
  'notes': [79, 78, 68, 75, 74, 62, 72, 71, 59, 68, 67]},
 {'id': 'M18',
  'name': 'min Hor 4',
  'direction': 3,
  'scale': 'min',
  'notes': [65, 60, 60, 62, 60, 56, 62, 60]},
 {'id': 'M19',
  'name': 'Eas Hor 2',
  'direction': 3,
  'scale': 'eas',
  'notes': [62, 60, 59, 60, 59, 62, 60, 59]},
 {'id': 'M20',
  'name': 'min Asc S 2',
  'direction': 0,
  'scale': 'min',
  'notes': [51, 55, 63, 60, 72, 67, 75, 79]},
 {'id': 'M21',
  'name': 'min Hor 2',
  'direction': 3,
  'scale': 'min',
  'notes': [60, 60, 62, 60, 65, 60, 62, 60]},
 {'id': 'M22',
  'name': 'min Osc 5',
  'direction': 2,
  'scale': 'min',
  'notes': [63, 55, 62, 55, 63, 55, 56, 55]},
 {'id': 'M23',
  'name': 'min Osc 4',
  'direction': 2,
  'scale': 'min',
  'notes': [56, 67, 58, 56, 65, 60, 67, 56]},
 {'id': 'M24',
  'name': 'min Des S 6',
  'direction': 1,
  'scale': 'min',
  'notes': [79, 72, 74, 67, 72, 63, 62, 60]},
 {'id': 'M25',
  'name': 'Maj Hor 6',
  'direction': 3,
  'scale': 'maj',
  'notes': [60, 62, 60, 57, 62, 60, 62, 60]},
 {'id': 'M26',
  'name': 'Maj Hor 4',
  'direction': 3,
  'scale': 'maj',
  'notes': [57, 60, 59, 60, 60, 60, 57, 60]},
 {'id': 'M27',
  'name': 'Maj Osc 4',
  'direction': 2,
  'scale': 'maj',
  'notes': [60, 53, 55, 57, 60, 55, 60, 57]},
 {'id': 'M28',
  'name': 'Maj Osc 1',
  'direction': 2,
  'scale': 'maj',
  'notes': [52, 60, 53, 60, 55, 60, 57, 55]},
 {'id': 'M29',
  'name': 'Maj Hor 2',
  'direction': 3,
  'scale': 'maj',
  'notes': [60, 60, 62, 60, 65, 60, 62, 60]},
 {'id': 'M30',
  'name': 'Maj Asc 1',
  'direction': 0,
  'scale': 'maj',
  'notes': [48, 55, 50, 52, 60, 62, 64, 67]},
 {'id': 'M31',
  'name': 'Maj Asc 6',
  'direction': 0,
  'scale': 'maj',
  'notes': [55, 60, 62, 64, 67, 72, 74, 76]},
 {'id': 'M32',
  'name': 'min Osc 1',
  'direction': 2,
  'scale': 'min',
  'notes': [55, 63, 56, 62, 55, 63, 56, 55]},
 {'id': 'M33',
  'name': 'min Des S 4',
  'direction': 1,
  'scale': 'min',
  'notes': [79, 77, 74, 72, 68, 67, 65, 60]},
 {'id': 'M34',
  'name': 'min Asc S 4',
  'direction': 0,
  'scale': 'min',
  'notes': [53, 56, 60, 65, 68, 72, 77, 80]},
 {'id': 'M35',
  'name': 'Maj Osc 6',
  'direction': 2,
  'scale': 'maj',
  'notes': [64, 55, 60, 57, 55, 64, 57, 64]},
 {'id': 'M36',
  'name': 'min Des S 3',
  'direction': 1,
  'scale': 'min',
  'notes': [77, 74, 72, 68, 67, 65, 60, 56]},
 {'id': 'M37',
  'name': 'min Hor 1',
  'direction': 3,
  'scale': 'min',
  'notes': [60, 60, 63, 60, 62, 60, 63, 60]},
 {'id': 'M38',
  'name': 'min Hor 5',
  'direction': 3,
  'scale': 'min',
  'notes': [60, 60, 60, 60, 67, 60, 62, 60]},
 {'id': 'M39',
  'name': 'Maj Des 6',
  'direction': 1,
  'scale': 'maj',
  'notes': [77, 72, 71, 69, 65, 60, 59, 57]},
 {'id': 'M40',
  'name': 'Maj Des 1',
  'direction': 1,
  'scale': 'maj',
  'notes': [72, 71, 69, 65, 60, 59, 57, 53]},
 {'id': 'M41',
  'name': 'min Asc S 5',
  'direction': 0,
  'scale': 'min',
  'notes': [55, 60, 62, 63, 67, 72, 74, 75]},
 {'id': 'M42',
  'name': 'Maj Asc 3',
  'direction': 0,
  'scale': 'maj',
  'notes': [43, 59, 55, 62, 57, 59, 67, 74]},
 {'id': 'M43',
  'name': 'Maj Des 3',
  'direction': 1,
  'scale': 'maj',
  'notes': [75, 74, 72, 68, 63, 62, 60, 56]},
 {'id': 'M44',
  'name': 'min Osc 6',
  'direction': 2,
  'scale': 'min',
  'notes': [62, 63, 56, 55, 63, 62, 56, 55]},
 {'id': 'M45',
  'name': 'min Hor 3',
  'direction': 3,
  'scale': 'min',
  'notes': [60, 63, 60, 62, 60, 55, 62, 60]},
 {'id': 'M46',
  'name': 'Maj Osc 3',
  'direction': 2,
  'scale': 'maj',
  'notes': [55, 62, 57, 62, 59, 67, 60, 59]},
 {'id': 'M47',
  'name': 'Maj Osc 5',
  'direction': 2,
  'scale': 'maj',
  'notes': [62, 54, 57, 62, 55, 62, 57, 62]},
 {'id': 'M48',
  'name': 'Maj Des 4',
  'direction': 1,
  'scale': 'maj',
  'notes': [77, 74, 72, 70, 65, 62, 60, 58]},
 {'id': 'M49',
  'name': 'min Osc 3',
  'direction': 2,
  'scale': 'min',
  'notes': [58, 67, 60, 58, 65, 60, 67, 58]},
 {'id': 'M50',
  'name': 'Eas Osc 1',
  'direction': 2,
  'scale': 'eas',
  'notes': [62, 55, 56, 59, 60, 53, 62, 55]},
 {'id': 'M51',
  'name': 'min Hor 6',
  'direction': 3,
  'scale': 'min',
  'notes': [65, 60, 60, 60, 56, 60, 62, 60]},
 {'id': 'M52',
  'name': 'min Asc S 6',
  'direction': 0,
  'scale': 'min',
  'notes': [56, 60, 62, 65, 68, 72, 74, 77]},
 {'id': 'M53',
  'name': 'Maj Hor 5',
  'direction': 3,
  'scale': 'maj',
  'notes': [60, 60, 60, 55, 62, 60, 62, 60]},
 {'id': 'M54',
  'name': 'Maj Asc 4',
  'direction': 0,
  'scale': 'maj',
  'notes': [45, 53, 48, 57, 53, 60, 65, 72]},
 {'id': 'M55',
  'name': 'Eas Asc 1',
  'direction': 0,
  'scale': 'eas',
  'notes': [55, 56, 60, 62, 67, 68, 72, 74]},
 {'id': 'M56',
  'name': 'min Asc S 3',
  'direction': 0,
  'scale': 'min',
  'notes': [50, 56, 62, 60, 68, 65, 72, 74]}]

INITIAL_NOTES = [{'direction': 0, 'scale': 'maj', 'initial_note': ['52', '48', '48', '55', '43', '45']},
 {'direction': 0, 'scale': 'min', 'initial_note': ['48', '51', '53', '55', '56', '50']},
 {'direction': 1, 'scale': 'maj', 'initial_note': ['79', '74', '77', '72', '75', '77']},
 {'direction': 1, 'scale': 'min', 'initial_note': ['75', '79', '80', '79', '79', '77']},
 {'direction': 2, 'scale': 'maj', 'initial_note': ['53', '60', '52', '64', '55', '62']},
 {'direction': 2, 'scale': 'min', 'initial_note': ['56', '63', '56', '55', '62', '58']},
 {'direction': 3, 'scale': 'maj', 'initial_note': ['60', '60', '60', '57', '60', '60']},
 {'direction': 3, 'scale': 'min', 'initial_note': ['65', '60', '60', '60', '60', '65']}]

MOTIFS = pd.DataFrame(TRAINING_MOTIFS)
ACTIVE_MOTIFS = MOTIFS[MOTIFS.scale.isin(["maj", "min"])].copy()
display(MOTIFS[["id", "name", "scale", "direction", "notes"]])
print("Active motifs:", len(ACTIVE_MOTIFS), "| Archived eas motifs:", sum(MOTIFS.scale == "eas"))


,id,name,scale,direction,notes
0,M01,min Asc S 1,min,0,"[48, 55, 60, 62, 63, 67, 72, 79]"
1,M02,Eas Asc 2,eas,0,"[55, 56, 53, 59, 60, 56, 62, 65, 59, 67, 68]"
2,M03,Maj Asc 5,maj,0,"[52, 53, 55, 60, 64, 65, 67, 72]"
3,M04,Eas Osc 2,eas,2,"[66, 63, 56, 59, 67, 55, 60, 59]"
4,M05,min Des S 2,min,1,"[75, 74, 72, 67, 63, 62, 60, 55]"
5,M06,min Des S 1,min,1,"[79, 75, 74, 72, 67, 63, 62, 60]"
6,M07,Eas Hor 1,eas,3,"[55, 56, 59, 55, 56, 59, 56, 55]"
7,M08,Maj Des 5,maj,1,"[79, 74, 72, 71, 67, 62, 60, 59]"
8,M09,Maj Osc 2,maj,2,"[53, 60, 55, 60, 57, 60, 55, 53]"
9,M10,Eas Des 1,eas,1,"[79, 78, 75, 74, 72, 71, 68, 67]"


Active motifs: 48 | Archived eas motifs: 8


### Portfolio evidence — your training-material score
Run the next cell to create **one PNG containing all 48 active motifs, one after another**, with labels and boundaries. All pitch lists are shown in full. Equal quarter notes and sharp spellings are display conventions; this is not a reconstruction of the original MIDI scores.

Save the PNG using the following download cell (or the Colab Files panel). You do not need to insert it into this notebook. Add a 60–100 word caption to your portfolio: identify the dataset, explain the display conventions, and describe one pattern you notice. Listen to a selected motif afterwards and compare it with its notation.

**Your figure caption (60–100 words):**

[Identify the data, explain the notation conventions and describe one musical pattern you notice.]


In [4]:
# Install the Python library for creating and processing musical scores.
%pip -q install music21

import shutil
import subprocess
from music21 import environment

# Install MuseScore and the tools required to run it without a physical screen.
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "musescore3", "xvfb", "xauth"],
    check=True
)

# Find the MuseScore executable automatically.
MUSESCORE_PATH = next(
    (
        path
        for name in ["musescore3", "mscore3", "musescore", "mscore"]
        if (path := shutil.which(name))
    ),
    None
)

if MUSESCORE_PATH is None:
    raise RuntimeError("MuseScore executable was not found.")

if shutil.which("xvfb-run") is None:
    raise RuntimeError("The virtual screen tool xvfb-run was not found.")

# Configure music21 to use this MuseScore installation.
env = environment.Environment()
env["musescoreDirectPNGPath"] = MUSESCORE_PATH
env["musicxmlPath"] = MUSESCORE_PATH

print("MuseScore is ready:", MUSESCORE_PATH)

MuseScore is ready: /usr/bin/musescore3


In [5]:
import re
import subprocess
import tempfile
from pathlib import Path

from PIL import Image as PILImage
from music21 import (
    stream, note, metadata, clef, meter,
    expressions, layout, bar
)


def save_training_score(rows, filename="chromatone_training_notes.png"):
    """
    Render all training motifs with MuseScore.

    Each motif starts on a new system.
    All pitches are displayed as quarter notes because the training
    data contains no original note durations.

    MuseScore pages are combined into one downloadable PNG file.
    """
    rows = list(rows)

    if not rows:
        raise ValueError("No training motifs were provided.")

    if not globals().get("MUSESCORE_PATH"):
        raise RuntimeError("Run the MuseScore installation cell first.")

    output_path = Path(filename).resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Create a single part containing every motif in sequence.
    score = stream.Score()
    score.metadata = metadata.Metadata()
    score.metadata.title = "ChromaTone — Training Motifs"
    score.metadata.composer = (
        "Pitch-only examples • Equal quarter-note durations"
    )

    part = stream.Part()
    measure_number = 1

    # Use consistent sharp spellings for chromatic pitches.
    pitch_names = [
        "C", "C#", "D", "D#", "E", "F",
        "F#", "G", "G#", "A", "A#", "B"
    ]

    for motif_index, row in enumerate(rows):
        pitches = [int(value) for value in row["notes"]]

        if not pitches:
            raise ValueError(f"Motif {row['id']} contains no notes.")

        if any(value < 0 or value > 127 for value in pitches):
            raise ValueError(f"Motif {row['id']} contains invalid MIDI pitches.")

        direction = DIRECTIONS.get(
            int(row["direction"]), str(row["direction"])
        )
        label = (
            f"{row['id']} | {row['name']} | "
            f"{row['scale']} | {direction}"
        )

        # Split each motif into measures of up to four quarter notes.
        for offset in range(0, len(pitches), 4):
            chunk = pitches[offset:offset + 4]
            measure = stream.Measure(number=measure_number)
            measure_number += 1

            # A short final measure gets its actual duration.
            # Hidden time signatures avoid adding artificial rests.
            signature = meter.TimeSignature(f"{len(chunk)}/4")
            signature.style.hideObjectOnPrint = True
            measure.insert(0, signature)

            if offset == 0:
                if motif_index == 0:
                    measure.insert(0, clef.TrebleClef())
                else:
                    measure.insert(
                        0, layout.SystemLayout(isNew=True)
                    )

                heading = expressions.TextExpression(label)
                heading.placement = "above"
                measure.insert(0, heading)

            for midi_pitch in chunk:
                octave = midi_pitch // 12 - 1
                pitch_name = pitch_names[midi_pitch % 12]

                training_note = note.Note(
                    f"{pitch_name}{octave}",
                    quarterLength=1
                )
                measure.append(training_note)

            # Mark the boundary between training motifs.
            if offset + 4 >= len(pitches):
                measure.rightBarline = bar.Barline("double")

            part.append(measure)

    score.append(part)

    print(f"Rendering {len(rows)} training motifs with MuseScore…")

    with tempfile.TemporaryDirectory() as folder:
        folder = Path(folder)
        xml_path = folder / "training.musicxml"
        png_path = folder / "training.png"

        score.write("musicxml", fp=str(xml_path))

        result = subprocess.run(
            [
                "xvfb-run", "-a",
                MUSESCORE_PATH,
                "-s",                 # Disable the audio synthesizer.
                "-r", "150",          # Image resolution in DPI.
                "-o", str(png_path),
                str(xml_path),
            ],
            capture_output=True,
            text=True,
            timeout=180,
        )

        if result.returncode != 0:
            raise RuntimeError(
                "MuseScore export failed:\n"
                + result.stdout + "\n" + result.stderr
            )

        # Keep pages in numerical order: 1, 2, ..., 10, ...
        def page_number(path):
            match = re.search(r"-(\d+)$", path.stem)
            return int(match.group(1)) if match else 0

        page_paths = sorted(
            folder.glob("training*.png"),
            key=page_number
        )

        if not page_paths:
            raise RuntimeError(
                "MuseScore did not create any PNG files.\n"
                + result.stdout + "\n" + result.stderr
            )

        # Flatten transparent pages onto a white background.
        pages = []

        for page_path in page_paths:
            with PILImage.open(page_path) as source:
                rgba = source.convert("RGBA")
                page = PILImage.new("RGB", rgba.size, "white")
                page.paste(rgba, mask=rgba.getchannel("A"))
                pages.append(page)

        # Combine all score pages into one vertical image.
        gap = 24
        width = max(page.width for page in pages)
        height = sum(page.height for page in pages)
        height += gap * (len(pages) - 1)

        combined = PILImage.new("RGB", (width, height), "white")

        y = 0
        for page in pages:
            x = (width - page.width) // 2
            combined.paste(page, (x, y))
            y += page.height + gap

        combined.save(output_path, dpi=(150, 150))

    print(f"Combined {len(page_paths)} score pages into one PNG.")
    return output_path


score_path = save_training_score(
    ACTIVE_MOTIFS.to_dict("records")
)

print(
    "Saved:", score_path,
    "— all", len(ACTIVE_MOTIFS), "active motifs"
)

Rendering 48 training motifs with MuseScore…
Combined 5 score pages into one PNG.
Saved: /content/chromatone_training_notes.png — all 48 active motifs


In [6]:
# Download the score. This cell requires Google Colab.
from google.colab import files
files.download(str(score_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
def synthesize(notes, root_index, seconds_per_note=0.24, sample_rate=22050):
    """Render pitches with a simple plucked sound and root accompaniment."""
    pieces = []
    root_hz = 440.0 * 2 ** ((48 + root_index - 69) / 12)
    for midi_note in notes:
        duration = float(np.clip(seconds_per_note, 0.08, 0.6))
        n = max(1, int(sample_rate * duration))
        t = np.arange(n) / sample_rate
        freq = 440.0 * 2 ** ((int(midi_note) - 69) / 12)
        # A decaying envelope and harmonics shape each note.
        envelope = np.exp(-5.0 * t / duration)
        voice = (np.sin(2*np.pi*freq*t) + 0.22*np.sin(4*np.pi*freq*t) + 0.08*np.sin(6*np.pi*freq*t)) * envelope
        drone = 0.12 * np.sin(2*np.pi*root_hz*t) * np.minimum(1, t*30) * np.minimum(1, (duration-t)*30)
        pieces.append((voice * 0.72 + drone).astype(np.float32))
        pieces.append(np.zeros(int(sample_rate * 0.025), dtype=np.float32))
    audio = np.concatenate(pieces) if pieces else np.zeros(1, dtype=np.float32)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak * 0.85
    return Audio(audio, rate=sample_rate)


MOTIF_TO_HEAR = "M01"  # Change to any ID in the table.
selected = MOTIFS[MOTIFS.id == MOTIF_TO_HEAR].iloc[0]
print(selected["name"], selected["notes"])
display(synthesize(selected["notes"], root_index=0))


min Asc S 1 [48, 55, 60, 62, 63, 67, 72, 79]


## 4. Markov chains: learn continuations from examples


### Portfolio task — How does a Markov chain work?

Watch those two Videos. Not THAT easy but worth it :-)

https://www.youtube.com/watch?v=gn-_ocUaGYo

https://www.youtube.com/watch?v=V7OPB6zmSdM
( the code is here in a notebook file, instead of a .py file: https://github.com/Eratofee/ChromaTone/blob/main/Markov_Chain_Example.ipynb )


Before running the next code cell, research the following question. Write again **120–180 words** in your own words. Record at least one source with author/organisation, title, URL and access date. Distinguish a source’s claim from your own interpretation.

Explain states, transitions and transition probabilities. What distinguishes a first-order chain from a second-order chain? Use a three-note example to explain the latter. Describe one strength and one limitation for melody generation. Research first; then compare your explanation with the implementation below.

Double-click the following text cell to edit it. Keep your completed text for your portfolio.


**Your response:**

[Write your researched explanation here.]

**Sources:**

[Add your references here.]

**Your own question or design idea:**

[Write here.]


### 4a. Training = counting observed continuations
Our state is a pair of pitches. For every three consecutive entries, store the third as a possible continuation of the first two. Repeated continuations stay in the list: `[67, 67, 69]` gives 67 probability 2/3 and 69 probability 1/3.

We prepend `-1` as a start marker (not a playable note), and never join one training motif to the next. Major/minor and direction select the training subset **before** transitions are counted.


In [8]:
def train_markov(sequences):
    transitions = {}
    for notes in sequences:
        sequence = [-1] + list(notes)
        for a, b, following in zip(sequence, sequence[1:], sequence[2:]):
            transitions.setdefault((int(a), int(b)), []).append(int(following))
    return transitions

# Small, readable example; this does not change the instrument's dataset.
toy = train_markov([[60,64,67], [60,64,67], [60,64,69]])
for state, options in toy.items():
    print(state, '→', {n: round(c/len(options),3) for n,c in Counter(options).items()})


(-1, 60) → {64: 1.0}
(60, 64) → {67: 0.667, 69: 0.333}


### 4b. Generation = weighted random selection
Select a valid starting pair, choose a continuation, shift the pair and repeat. A dead end restarts from a valid beginning. The live instrument keeps its pair across phrases and resets when the selected model changes.

The seed pitch in `(-1, first_pitch)` is context: this implementation emits the selected continuation first. It does not explicitly emit that seed pitch.

The chain learns from the embedded pitch examples, **not from your painting**. Drawing chooses settings and subsets. Changing the tonal centre transposes pitches afterwards. Labels such as “major” describe the dataset category; there is no additional scale correction.


In [9]:
LIVE_STATE = {'model': None, 'pair': None}

def reset_chain():
    LIVE_STATE.update(model=None, pair=None)

def select_model(scale, direction):
    rows = ACTIVE_MOTIFS[(ACTIVE_MOTIFS.scale == scale) & (ACTIVE_MOTIFS.direction == direction)]
    if rows.empty:
        rows = ACTIVE_MOTIFS[(ACTIVE_MOTIFS.scale == scale) & (ACTIVE_MOTIFS.direction == 3)]
    if rows.empty: raise ValueError('No matching training material.')
    return train_markov(rows.notes)

def initial_pair(transitions, scale, direction, rng):
    entry = next((r for r in INITIAL_NOTES if r['scale']==scale and r['direction']==direction), None)
    candidates = [int(n) for n in entry['initial_note'] if (-1,int(n)) in transitions] if entry else []
    if not candidates: candidates = [b for a,b in transitions if a == -1]
    if not candidates: raise ValueError('No valid start state.')
    return (-1, rng.choice(candidates))

def generate_notes(scale, direction, amount=6, rng=None, state=None):
    rng = rng or random
    state = LIVE_STATE if state is None else state
    transitions = select_model(scale, direction)
    key = (scale, direction)
    pair = state.get('pair')
    if state.get('model') != key or pair not in transitions:
        pair = initial_pair(transitions, scale, direction, rng)
    result = []
    for _ in range(amount):
        if pair not in transitions:
            pair = initial_pair(transitions, scale, direction, rng)
        following = rng.choice(transitions[pair])
        result.append(following)
        pair = (pair[1], following)
    state.update(model=key, pair=pair)
    return result

example_model = select_model('maj', 0)
example_state = next(s for s in example_model if s[0] != -1)
print('A real state:', example_state, 'Possible continuations:', example_model[example_state])
for seed in (1, 2, 3):
    print('Seed', seed, generate_notes('maj', 0, 12, rng=random.Random(seed), state={}))


A real state: (52, 53) Possible continuations: [55]
Seed 1 [57, 53, 55, 60, 65, 72, 53, 48, 57, 53, 55, 60]
Seed 2 [53, 55, 60, 64, 65, 67, 72, 74, 76, 60, 62, 64]
Seed 3 [57, 53, 60, 65, 67, 72, 74, 76, 55, 50, 52, 60]


**Pause and explain:** Why can a repeated transition be more likely? Why might two runs still sound identical? Describe what happens at a dead end. Add 2–3 sentences below after examining the output.


**My observations:**

[Write here.]


## 5. Measure the drawing and connect the rules


### Portfolio task — What does the system actually see?

Before running the next code cell, research the following question. Write **120–180 words** in your own words ... . Record at least one source with author/organisation, title, URL and access date. Distinguish a source’s claim from your own interpretation.

Research hue, saturation and brightness. Explain why lighting, background and the chosen colour space matter for image-to-music mapping. For a real painting photographed by a camera, which current features could be reused and which would need an additional method?

Double-click the following text cell to edit it. Keep your completed text for your portfolio.


**Your response:**

[Write your researched explanation here.]

**Sources:**

[Add your references here.]

**Your own question or design idea:**

[Write here.]


In [10]:
def image_to_rgb(data_url):
    return np.asarray(Image.open(io.BytesIO(base64.b64decode(data_url.split(',',1)[1]))).convert('RGB'))

def image_features(rgb):
    hsv = rgb_to_hsv(np.asarray(rgb, dtype=float)/255)
    hue = np.minimum(np.rint(hsv[:,:,0]*180), 179)
    saturation, value = hsv[:,:,1]*255, hsv[:,:,2]*255
    black = value < 25
    white = (value > 204) & (saturation < 25)
    counts = {'white': int(white.sum())}
    for colour, ranges in COLOR_RANGES.items():
        mask = np.zeros(hue.shape, dtype=bool)
        for lo,hi in ranges: mask |= (hue >= lo) & (hue <= hi)
        counts[colour] = int((mask & ~black & ~white).sum())
    pitch_counts = np.zeros(12)
    for colour,count in counts.items(): pitch_counts[PITCH_CLASSES.index(COLOR_TO_NOTE[colour])] += count
    key = int(np.argmax(pitch_counts))
    mode = 'min' if value.mean() < BRIGHTNESS_THRESHOLD else 'maj'
    return key, mode, counts, float(value.mean())

def note_duration(speed):
    for threshold, duration in [(7000,.05),(5000,.07),(3000,.10),(1000,.15),(500,.25),(100,.30)]:
        if speed > threshold: return float(np.clip(duration*DURATION_MULTIPLIER,.08,.6))
    return float(np.clip(.35*DURATION_MULTIPLIER,.08,.6))

def drawing_to_music(data_url, direction, speed, active_colour, colour_hex):
    try:
        key, mode, counts, brightness = image_features(image_to_rgb(data_url))
        if active_colour:
            brush = np.array([[list(bytes.fromhex(colour_hex.lstrip('#')))]],dtype=np.uint8)
            brush_key, _, brush_counts, _ = image_features(brush)
            if sum(brush_counts.values()): key = brush_key
        if MODE_OVERRIDE != 'auto': mode = MODE_OVERRIDE
        notes = generate_notes(mode, int(direction), NOTES_PER_PHRASE)
        notes = np.clip(np.array(notes)+key+REGISTER_SHIFT,0,127).tolist()
        return DisplayJSON({'ok':True, 'notes':notes, 'root':48+key,
          'duration':note_duration(float(speed)),
          'summary':f'{PITCH_CLASSES[key]} | {mode} | {DIRECTIONS[int(direction)]} | brightness {brightness:.0f}'})
    except Exception as exc:
        return DisplayJSON({'ok':False, 'error':str(exc)})

print('Image analysis and music mapping ready.')


Image analysis and music mapping ready.


## 6. Play your instrument
Run the next two cells in Colab. Draw first, then press **Start live**. Allow browser sound if asked. Stop playback before rerunning cells or changing settings.

The displayed status reports the selected tonal centre, mode, direction and brightness. Recent pointer history remains until you draw more or clear the canvas. The minimum update interval is adjustable; network and phrase timing add delay. A phrase may repeat while waiting for new data. This is an interactive prototype, not a sample-accurate performance system.

The next cell is browser plumbing; you do not need to understand or edit it to complete the lab.


In [11]:
from google.colab import output
output.register_callback('chromatone.live', drawing_to_music)
print('Instrument connected. Run the canvas cell.')


Instrument connected. Run the canvas cell.


In [12]:
live_canvas_html = r'''
<div style="font-family:sans-serif; max-width:850px;">
  <div style="display:flex; flex-wrap:wrap; align-items:center; gap:12px; margin:8px 0;">
    <label>Colour <input id="ctlive-color" type="color" value="#ffffff"></label>
    <label>Brush size
      <input id="ctlive-size" type="range" min="2" max="36" value="10">
    </label>
    <label>
      <input id="ctlive-active" type="checkbox">
      Use selected colour as tonal centre
    </label>
    <button onclick="ctLiveClear()">Clear canvas</button>
    <button onclick="ctLiveStart()">Start live</button>
    <button onclick="ctLiveStop()">Stop live</button>
  </div>

  <canvas id="ctlive-canvas" width="800" height="420"
    style="width:100%; max-width:800px; height:auto; background:#000;
           border:1px solid #888; touch-action:none; cursor:crosshair;">
  </canvas>

  <div id="ctlive-status" style="margin:8px 0; color:#444;">
    Draw and click Start live.
  </div>
</div>

<script>
(function() {
  if (window.ctLiveStop) window.ctLiveStop();
  let generation = 0;
  const canvas = document.getElementById('ctlive-canvas');
  const ctx = canvas.getContext('2d');
  const status = document.getElementById('ctlive-status');

  ctx.fillStyle = '#000000';
  ctx.fillRect(0, 0, canvas.width, canvas.height);

  let drawing = false;
  let lastPoint = null;
  let directions = [];
  let speeds = [];

  let live = false;
  let busy = false;
  let pendingMusic = null;
  let currentMusic = null;
  let noteIndex = 0;
  let audioContext = null;
  let pollTimer = null;
  let playerTimer = null;

  function position(event) {
    const rect = canvas.getBoundingClientRect();
    return {
      x: (event.clientX - rect.left) * canvas.width / rect.width,
      y: (event.clientY - rect.top) * canvas.height / rect.height
    };
  }

  canvas.addEventListener('pointerdown', event => {
    drawing = true;
    lastPoint = {...position(event), time: performance.now()};
    canvas.setPointerCapture(event.pointerId);
  });

  canvas.addEventListener('pointermove', event => {
    if (!drawing || !lastPoint) return;

    const point = position(event);
    const now = performance.now();

    ctx.strokeStyle = document.getElementById('ctlive-color').value;
    ctx.lineWidth = Number(document.getElementById('ctlive-size').value);
    ctx.lineCap = 'round';
    ctx.lineJoin = 'round';
    ctx.beginPath();
    ctx.moveTo(lastPoint.x, lastPoint.y);
    ctx.lineTo(point.x, point.y);
    ctx.stroke();

    const dy = point.y - lastPoint.y;
    directions.push(dy < 0 ? 0 : (dy > 0 ? 1 : 3));
    if (directions.length > 100) directions.shift();

    const distance = Math.hypot(point.x - lastPoint.x, point.y - lastPoint.y);
    const elapsed = (now - lastPoint.time) / 1000;
    if (elapsed > 0) {
      speeds.push(distance / elapsed);
      if (speeds.length > 100) speeds.shift();
    }

    lastPoint = {...point, time: now};
  });

  canvas.addEventListener('pointercancel', () => { drawing = false; lastPoint = null; });
  canvas.addEventListener('pointerup', () => {
    drawing = false;
    lastPoint = null;
  });

  function currentTrend() {
    if (directions.length === 0) return 3;

    const up = directions.filter(x => x === 0).length;
    const down = directions.filter(x => x === 1).length;
    const flat = directions.filter(x => x === 3).length;

    if (Math.abs(up - down) < 10 && flat < 10) return 2;
    if (up > directions.length / 2) return 0;
    if (down > directions.length / 2) return 1;
    if (flat > directions.length / 2) return 3;
    return 2;
  }

  function currentSpeed() {
    if (speeds.length === 0) return 0;
    return speeds.reduce((sum, value) => sum + value, 0) / speeds.length;
  }

  function playNote(midiNote, rootNote, duration) {
    if (!audioContext) return;

    const start = audioContext.currentTime;
    const length = Math.max(0.08, duration);

    function pluck(frequency, volume, decay) {
      const oscillator = audioContext.createOscillator();
      const gain = audioContext.createGain();

      oscillator.type = 'sine';
      oscillator.frequency.value = frequency;
      gain.gain.setValueAtTime(0.001, start);
      gain.gain.linearRampToValueAtTime(volume, start + 0.01);
      gain.gain.exponentialRampToValueAtTime(0.001, start + decay);

      oscillator.connect(gain);
      gain.connect(audioContext.destination);
      oscillator.start(start);
      oscillator.stop(start + decay + 0.02);
    }

    const melodyHz = 440 * 2 ** ((midiNote - 69) / 12);
    const rootHz = 440 * 2 ** ((rootNote - 69) / 12);

    pluck(melodyHz, 0.16, length);
    pluck(melodyHz * 2, 0.035, length * 0.7);
    pluck(rootHz, 0.045, length);
  }

  function scheduleNextNote() {
    if (!live) return;

    if (pendingMusic && (!currentMusic || noteIndex >= currentMusic.notes.length)) {
      currentMusic = pendingMusic;
      pendingMusic = null;
      noteIndex = 0;
      status.textContent = currentMusic.summary;
    }

    if (!currentMusic || !currentMusic.notes.length) {
      playerTimer = setTimeout(scheduleNextNote, 100);
      return;
    }

    if (noteIndex >= currentMusic.notes.length) noteIndex = 0;

    const note = currentMusic.notes[noteIndex++];
    playNote(note, currentMusic.root, currentMusic.duration);
    playerTimer = setTimeout(
      scheduleNextNote,
      Math.max(60, currentMusic.duration * 1000)
    );
  }

  async function requestUpdate() {
    if (!live) return;


    if (busy || pendingMusic) {
      pollTimer = setTimeout(requestUpdate, 200);
      return;
    }

    busy = true;
    const token = generation;
    try {
      const result = await google.colab.kernel.invokeFunction(
        'chromatone.live',
        [
          canvas.toDataURL('image/png'),
          currentTrend(),
          currentSpeed(),
          document.getElementById('ctlive-active').checked,
          document.getElementById('ctlive-color').value
        ],
        {}
      );

      const payload = result?.data?.['application/json'];
      if (!payload || !payload.ok) {
        throw new Error(payload?.error || 'No music data received.');
      }

      if (live && token === generation) pendingMusic = payload;
    } catch (error) {
      status.textContent = 'Live analysis error: ' +
        (error?.message || String(error));
    } finally {
      if (token === generation) busy = false;
      if (live && token === generation) pollTimer = setTimeout(requestUpdate, __UPDATE_MS__);
    }
  }

  window.ctLiveStart = async function() {
    if (live) return;

    try {
      const AudioContextClass = window.AudioContext || window.webkitAudioContext;
      audioContext = audioContext || new AudioContextClass();
      await audioContext.resume();

      live = true;
      status.textContent = 'Live started — analysing the canvas…';

      if (!playerTimer) scheduleNextNote();
      requestUpdate();
    } catch (error) {
      status.textContent = 'Could not start audio: ' +
        (error?.message || String(error));
    }
  };

  window.ctLiveStop = function() {
    live = false;
    generation++;
    if (audioContext) audioContext.suspend();
    clearTimeout(pollTimer);
    clearTimeout(playerTimer);
    pollTimer = null;
    playerTimer = null;
    busy = false;
    pendingMusic = null;
    currentMusic = null;
    noteIndex = 0;
    status.textContent = 'Live playback stopped.';
  };

  window.ctLiveClear = function() {
    ctx.fillStyle = '#000000';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    directions = [];
    speeds = [];
    status.textContent = live
      ? 'Canvas cleared — live playback continues.'
      : 'Canvas cleared.';
  };
})();
</script>
'''

display(HTML(live_canvas_html.replace('__UPDATE_MS__', str(int(UPDATE_SECONDS * 1000)))))


## 7. Listening experiments — make the instrument yours
Do not write new code. Change the supplied values, run the experiment and compare what you hear.

### Experiment A: major and minor
Keep direction, phrase length, seed, register, root and duration fixed. Change only `LISTEN_MODE` from `maj` to `min` and rerun. This compares two different training subsets, **not** the same melody converted between modes. The fixed seed controls randomness but does not force identical melodic contours.

Record your impression before reading a partner’s description. Describe tension, contour, familiarity or texture rather than relying only on “happy/sad”.


In [15]:
LISTEN_MODE = 'maj'    # Compare 'maj' and 'min'.
LISTEN_DIRECTION = 2   # 0 ascending, 1 descending, 2 oscillating, 3 horizontal
LISTEN_SEED = 42
LISTEN_ROOT = 0        # 0=C, 1=Db, ... 11=B
LISTEN_REGISTER = 0    # Try -12 or 12 after the mode comparison.
LISTEN_DURATION = 0.24

assert LISTEN_MODE in ('maj','min') and LISTEN_DIRECTION in DIRECTIONS
assert 0 <= LISTEN_ROOT <= 11
comparison_notes = generate_notes(LISTEN_MODE, LISTEN_DIRECTION, 24,
                                  rng=random.Random(LISTEN_SEED), state={})
comparison_notes = np.clip(np.array(comparison_notes)+LISTEN_ROOT+LISTEN_REGISTER,0,127).tolist()
print(comparison_notes)
display(synthesize(comparison_notes, LISTEN_ROOT, LISTEN_DURATION))


[54, 57, 62, 59, 67, 60, 59, 54, 57, 62, 59, 67, 60, 59, 62, 57, 62, 59, 67, 60, 59, 53, 60, 55]


### Experiment B: live mappings
Return to section 2. Compare `MODE_OVERRIDE = "maj"` with `"min"` on the same drawing, then restore `"auto"`. Compare a sparse drawing with a nearly filled canvas. Next test one other variable: threshold, register, duration multiplier or phrase length.

Stop playback before each change; rerun the settings cell and start again. To change update timing, also rerun the canvas cell, which clears its drawing. Capture screenshots before doing so.

| Trial | Single changed value | What remained fixed? | Expected effect | What you heard | Artistic preference and reason |
|---|---|---|---|---|---|
| A | Major → minor | [Fill in] | [Fill in] | [Fill in] | [Fill in] |
| B | [Fill in] | [Fill in] | [Fill in] | [Fill in] | [Fill in] |
| C | [Fill in] | [Fill in] | [Fill in] | [Fill in] | [Fill in] |

**Write 120–180 words:** How did your experience change? Which result contradicted your prediction? What can you attribute to the changed parameter, and what might be due to randomness?


**My listening reflection:**

[Write here.]


### Optional: add one composed motif
Edit only the list and labels below. Use pitches relative to the existing C-based material so transposition remains meaningful. You can use a MIDI keyboard or a pitch chart to find note numbers; no new programming is required. Stop playback first. Rerunning replaces the same student motif rather than duplicating it.

After adding a motif, rerun the score-export cell to include it in your portfolio image. Compare output before/after and explain why a new motif may have little audible effect if its transitions rarely connect to existing states.


In [16]:
ADD_MY_MOTIF = False    # Set True when your motif is ready.
MY_NOTES = [60, 62, 64, 67, 64, 62, 60, 60]
MY_MODE = 'maj'
MY_DIRECTION = 2
MY_NAME = 'My first motif'

if ADD_MY_MOTIF:
    assert MY_MODE in ('maj','min') and MY_DIRECTION in DIRECTIONS
    assert len(MY_NOTES) >= 3 and all(isinstance(n,int) and 0 <= n <= 127 for n in MY_NOTES)
    new_row = {'id':'STUDENT', 'name':MY_NAME, 'scale':MY_MODE, 'direction':MY_DIRECTION, 'notes':MY_NOTES}
    MOTIFS = pd.concat([MOTIFS[MOTIFS.id != 'STUDENT'], pd.DataFrame([new_row])], ignore_index=True)
    ACTIVE_MOTIFS = MOTIFS[MOTIFS.scale.isin(['maj','min'])].copy()
    reset_chain()
    print('Student motif added/replaced. Active motifs:',len(ACTIVE_MOTIFS))
else:
    print('Optional activity skipped; existing material retained.')


Optional activity skipped; existing material retained.


## 8. Wednesday: performance and portfolio
Prepare a 2–3 minute demonstration with a clear beginning, one audible transformation and an ending. Decide who draws, who explains, who monitors sound and how the remote student participates. Test shared audio in advance. Keep a short recording as a fallback.

Your portfolio should include:
- Your own researched sections and source references.
- The training-note PNG with a caption explaining its limitations.
- Your experiment table, screenshots and listening reflection.
- Your proposed extra mappings and a justified choice of one to develop.
- A short performance record and a clear description of your personal contribution.

**Final reflection — 150–200 words:** Where is creativity located: in the examples, the mapping, the random choices or the performer? What did you decide? What would you change next, and why?

**Your reflection:**

[Write here.]

**Your contribution:**

[Write here.]

**Next development idea:**

[Write here.]

### First-version implementation notes
This teaching version preserves the supplied pitch data, colour mapping and second-order live-generation approach, but separates them into learning steps. It uses a self-contained image-analysis implementation with the same hue-bin scheme; boundary rounding may differ slightly from the OpenCV implementation. Note durations are consistently limited to 0.08–0.60 seconds. Research responses are deliberately left blank. External MIDI and camera capture are not implemented in this version.
